### 《实用Python编程》教学代码
## 第7章 大模型辅助编程

+ 本章所有提示词存放于 `../pybook-data/ch7/` 目录
+ 独立脚本（Streamlit应用、爬虫等）存放于 `../pycode/` 目录


## 7.2 利用大模型生成代码

提示词四要素（表7.2）：**角色设定、任务描述、约束与要求、输入输出示例**


### 示例代码7.1 猜数字游戏（人猜电脑想）

生成提示词见 [prompt-7-3.txt](../pybook-data/ch7/prompt-7-3.txt)。运行后输入 1~1000 之间的整数反复猜测：

In [ ]:
# 猜数字游戏（人猜电脑想）
import random
target = random.randint(1, 1000)   # 电脑想的数
tries = 0                          # 计数器
while True:
    guess = int(input("请输入 1-1000 之间的整数："))
    tries += 1
    if guess == target:
        print(f"恭喜，猜对了！共猜了 {tries} 次。")
        break
    print("大了" if guess > target else "小了")

### 示例代码7.2 / 7.3 网页版猜数字游戏（Streamlit）

完整代码见 [code-7-2.py](../pycode/code-7-2.py)（保存为 `guess-app.py` 后执行 `streamlit run guess-app.py`）。

7.2 存在缺陷：第一局结束后点击"再来一局"无法重启游戏（`target` 未重新生成）。将报错现象反馈给大模型后得到修复版 [code-7-3-bugfix.py](../pycode/code-7-3-bugfix.py)——这正是"多轮对话迭代优化"的过程。

### 示例代码7.4 网页数据爬虫

生成提示词见 [prompt-7-4.txt](../pybook-data/ch7/prompt-7-4.txt)；完整代码见 [code-7-4.py](../pycode/code-7-4.py)，保存为 `crawler.py` 后运行 `python crawler.py`。

爬取结果已存于 [gaokao_page_1.html](../pybook-data/ch7/gaokao_page_1.html)（投档线）与 [gaokao_page_2.html](../pybook-data/ch7/gaokao_page_2.html)（分段表）。

### 示例代码7.5 网页数据清洗

生成提示词见 [prompt-7-5.txt](../pybook-data/ch7/prompt-7-5.txt)。先清洗分段表：

In [1]:
import pandas as pd
# 读取并清洗分段表
df_score = pd.read_html('../pybook-data/ch7/gaokao_page_2.html')[0].iloc[1:].reset_index(drop=True)
df_score.columns = ['分数', '人数', '位次']
# 定位并删除本科线之后的行（假设本科线行包含文本标注）
undergrad_idx = df_score[df_score['分数'].astype(str).str.contains('本科线')].index
if len(undergrad_idx):
    df_score = df_score.iloc[:undergrad_idx[0]+1]
# 提取分数数字，处理第一行特殊情况
df_score['分数'] = df_score['分数'].astype(str).str.extract(r'(\d+)')[0]
df_score.loc[0, '分数'] = '700'  # 将"700-750"简化为"700"
df_score['分数'] = pd.to_numeric(df_score['分数'])
# 创建分数-位次映射字典
score_to_rank = dict(zip(df_score['分数'], df_score['位次']))


print(df_score.head())
print(f"分段表：{df_score.shape[0]} 行")

    分数   人数   位次
0  700  117  117
1  699   15  132
2  698   16  148
3  697   17  165
4  696   18  183
分段表：267 行


再清洗投档线表，并用分段表的"分数→位次"映射为每个投档线换算位次：

In [1]:
import pandas as pd

# 创建分数-位次映射
score_to_rank = dict(zip(df_score['分数'], df_score['位次']))
# 读取并清洗投档线表
df_admission = pd.read_html('../pybook-data/ch7/gaokao_page_1.html')[0].iloc[1:].reset_index(drop=True)
df_admission.columns = ['院校代码', '院校', '专业组代码', '限制', '投档线']
df_admission['投档线'] = pd.to_numeric(df_admission['投档线'].astype(str).str.extract(r'(\d+)')[0])
# 添加位次列（通过映射字典）
df_admission['位次'] = df_admission['投档线'].map(score_to_rank)
# 输出CSV文件
df_score.to_csv('cleaned_score_segments.csv', index=False, encoding='utf-8-sig')
df_admission.to_csv('cleaned_admission_lines.csv', index=False, encoding='utf-8-sig')

print(df_admission.head())
print(f"投档线表：{df_admission.shape[0]} 行")

   院校代码    院校 专业组代码     限制  投档线   位次
0  1021  北京大学     2     不限  700  117
1  1023  清华大学     2     物理  698  148
2  1021  北京大学     3  物理＋化学  690  354
3  1023  清华大学     3  物理＋化学  689  384
4  1021  北京大学     1     不限  688  423
投档线表：1258 行


### 示例代码7.6～7.11 高考志愿填报推荐系统（Streamlit多模块项目）

生成提示词见 [prompt-7-6.txt](../pybook-data/ch7/prompt-7-6.txt)。完整项目见 [ch7-gaokao-recommender/](../pycode/ch7-gaokao-recommender/)：

```text
ch7-gaokao-recommender/
├── main.py           # Streamlit入口，页面布局与逻辑协调
├── config.py         # 配置参数（人数比例、区间系数等）
├── data_loader.py    # 数据加载与预处理
├── calculator.py     # 位次调整与区间计算逻辑
├── recommender.py    # 推荐算法实现
└── ui_components.py  # UI组件渲染函数
```

将清洗得到的 `cleaned_admission_lines.csv` 放入项目目录后，执行 `streamlit run main.py` 启动系统。

## 7.3 利用大模型测试与调试代码

| | 代码测试 | 代码调试 |
|---|---|---|
| 目标 | 发现问题（找出bug） | 解决问题（消除bug） |
| 性质 | 结构化、预防性 | 探索性、诊断性 |
| 工具 | pytest | pdb |

### 示例代码7.12 基于Zeller公式的日期转星期程序（带有隐患）

测试提示词见 [prompt-7-8.txt](../pybook-data/ch7/prompt-7-8.txt)。将函数保存为模块文件：

In [1]:
%%writefile ch7-demo/mymodule.py
def date_to_weekday(date_str):
    try:
        year, month, day = map(int, date_str.split('-'))
    except:
        return "Invalid date format. Please use 'YYYY-MM-DD'."
    if month < 3:
        year -= 1
        month += 12 #1月和2月变为上一年的13月和14月
    q = day
    m = month
    k = year % 100
    j = year // 100
    h = (q + (13 * (m + 1)) // 5 + k + k // 4 + j // 4 + 5 * j) % 7
    weekdays = ["Saturday", "Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
    return weekdays[h]

Writing mymodule.py


快速检查：合法日期结果正确，但非法日期 `1900-02-30`（2月没有30号）仍返回了一个"看似合法"的星期——隐患暴露：

In [1]:
from mymodule import date_to_weekday   # 从ch7-demo/mymodule.py导入

print(date_to_weekday("2024-08-29"))    # 合法日期
print(date_to_weekday("1900-02-30"))   # 非法日期：2月没有30号！
print(date_to_weekday("2000-02-29"))   # 闰年2月29日

Thursday
Friday
Tuesday


### 示例代码7.13 大模型生成的pytest测试用例

覆盖正确性对照、边界、闰年、月/日越界、格式错误、非法输入等场景（节选）：

In [1]:
%%writefile ch7-demo/test_date_weekday.py
import pytest
from mymodule import date_to_weekday  # 被测函数
# 工具：已知正确结果对照表（可再扩充）
KNOWN_DATES = {
    "1900-01-01": "Monday",
    "1900-12-31": "Monday",
    "1999-12-31": "Friday",
    "2000-02-29": "Tuesday",   # 闰年
    "2023-12-31": "Sunday",
    "9999-12-31": "Friday",
}
class TestDateToWeekday:
    """全面测试 date_to_weekday 函数"""
    # 1. 已知的正确性对照
    @pytest.mark.parametrize("date_str, expected", KNOWN_DATES.items())
    def test_known_dates(self, date_str, expected):
        assert date_to_weekday(date_str) == expected
    # 此处省略部分代码    
    # 6. 月份 / 天数越界
    @pytest.mark.parametrize("bad_date",
        [   "1900-13-01",  # 13 月
            "1900-00-01",  # 0 月
            "1900-01-32",  # 32 日
            "1900-01-00",  # 0 日
            "1900-02-30",  # 非闰年 2-30
            "2023-04-31",  # 4 月只有 30 天
        ],)
    def test_out_of_range_date_returns_wrong_weekday(self, bad_date):
        # 当前实现不会抛错，但结果必错；可断言与任何正确星期都不匹配
        result = date_to_weekday(bad_date)
        assert result not in KNOWN_DATES.values()        
    # 7. 格式错误
    @pytest.mark.parametrize("bad_format",
        [   "23-06-15",      # 年不是 4 位
            "2023/06/15",    # 分隔符错
            "2023-6-15",     # 月不是 2 位
            "2023-06-5",     # 日不是 2 位
            "2023-06",       # 缺日
            "hello",         # 完全非法
            "",              # 空串
        ],  )
    def test_invalid_format_returns_error_message(self, bad_format):
        assert date_to_weekday(bad_format).startswith("Invalid date format")
    # 此处省略部分代码

Writing test_date_weekday.py


In [1]:
!cd ch7-demo && python -m pytest -q test_date_weekday.py

......FF.FFFF.FF...                                                      [100%]
=================================== FAILURES ===================================
__ TestDateToWeekday.test_out_of_range_date_returns_wrong_weekday[1900-13-01] __

self = <test_date_weekday.TestDateToWeekday object at 0x108a32250>
bad_date = '1900-13-01'

    @pytest.mark.parametrize("bad_date",
        [   "1900-13-01",  # 13 月
            "1900-00-01",  # 0 月
            "1900-01-32",  # 32 日
            "1900-01-00",  # 0 日
            "1900-02-30",  # 非闰年 2-30
            "2023-04-31",  # 4 月只有 30 天
        ],)
    def test_out_of_range_date_returns_wrong_weekday(self, bad_date):
        # 当前实现不会抛错，但结果必错；可断言与任何正确星期都不匹配
        result = date_to_weekday(bad_date)
>       assert result not in KNOWN_DATES.values()
E       AssertionError: assert 'Tuesday' not in dict_values(['Monday', 'Monday', 'Friday', 'Tuesday', 'Sunday', 'Friday'])
E        +  where dict_values(['Monday', 'Monday', 'Friday', 'Tuesday', 'S

**8个用例失败**：月份/天数越界的日期没有被拒绝，反而算出了"看似合法"的星期几——示例代码7.12的隐患被测试捕获。

### 示例代码7.14 经大模型纠正后的日期转星期程序

将目标函数、测试用例与测试日志一并交给大模型，它给出的修复方案：用 `datetime.strptime` 一次性完成格式、闰年、月/日越界的全部校验：

In [1]:
%%writefile ch7-demo/mymodule.py
from datetime import datetime

_WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
def date_to_weekday(date_str: str) -> str:
    try:
        dt = datetime.strptime(date_str, "%Y-%m-%d")
        if dt.year < 1900:                       # 低于题目最小日期
            raise ValueError
    except Exception:
        return "Invalid date format. Please use 'YYYY-MM-DD'."

    # Zeller 公式（简化版）（1900-03-01 起可直接用，1/2 月按上年处理）
    y, m, d = dt.year, dt.month, dt.day
    if m < 3:
        y -= 1
        m += 12
    k, j = y % 100, y // 100
    h = (d + (13 * (m + 1)) // 5 + k + k // 4 + j // 4 + 5 * j) % 7
    # Zeller 原始映射 0=Saturday..6=Friday，与 _WEEKDAYS 顺序对齐
    return _WEEKDAYS[(h + 5) % 7]        # +5 偏移使 0->Monday..6->Sunday

Writing mymodule.py


In [1]:
!cd ch7-demo && python -m pytest -q test_date_weekday.py

..............FF...                                                      [100%]
=================================== FAILURES ===================================
____ TestDateToWeekday.test_invalid_format_returns_error_message[2023-6-15] ____

self = <test_date_weekday.TestDateToWeekday object at 0x1092f71d0>
bad_format = '2023-6-15'

    @pytest.mark.parametrize("bad_format",
        [   "23-06-15",      # 年不是 4 位
            "2023/06/15",    # 分隔符错
            "2023-6-15",     # 月不是 2 位
            "2023-06-5",     # 日不是 2 位
            "2023-06",       # 缺日
            "hello",         # 完全非法
            "",              # 空串
        ],  )
    def test_invalid_format_returns_error_message(self, bad_format):
>       assert date_to_weekday(bad_format).startswith("Invalid date format")
E       AssertionError: assert False
E        +  where False = <built-in method startswith of str object at 0x10937adf0>('Invalid date format')
E        +    where <built-in method startswith of str objec

**仅剩2个失败用例**：`2023-6-15` 与 `2023-06-5`。这两个日期本身合法，能通过 `datetime` 检查，属于测试用例"过严"——可在测试中注释掉，或按习题7.1引入更严格的格式校验。

> **教训**：不能对大模型给出的（测试）方案照单全收，要"以我为主"，根据实际任务需求判断方案的合理性。

## 7.4 利用大模型补全代码

### 示例代码7.15 注释引导法：先写注释，Copilot生成函数体

In [1]:
# 编写一个函数，实现对列表中的元素进行升序排序。排序算法采用冒泡排序。
# 函数输入为一个列表，输出为排序后的列表。
def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr

# 验证
print(bubble_sort([5, 1, 4, 2, 8]))

[1, 2, 4, 5, 8]


### 示例代码7.16 函数及参数名引导法：语义明确的命名即需求说明

In [1]:
def get_longest_common_substring_length(str1: str, str2: str) -> int:
    """Get the length of the longest common substring between two strings."""
    m = len(str1)
    n = len(str2)
    # Create a 2D array to store lengths of longest common suffixes of substrings
    LCSuff = [[0] * (n + 1) for _ in range(m + 1)]
    result = 0  # To store length of the longest common substring

    # Build LCSuff[m+1][n+1] in bottom-up fashion
    for i in range(m + 1):
        for j in range(n + 1):
            if i == 0 or j == 0:
                LCSuff[i][j] = 0
            elif str1[i - 1] == str2[j - 1]:
                LCSuff[i][j] = LCSuff[i - 1][j - 1] + 1
                result = max(result, LCSuff[i][j])
            else:
                LCSuff[i][j] = 0
    return result

# 验证
print(get_longest_common_substring_length("ABCBDAB", "BDCABA"))
print(get_longest_common_substring_length("hello", "yellow"))

2
4


### 示例代码7.17 测试案例引导法：先写测试用例，Copilot反推实现

In [1]:
def test_merge_dicts():
    dict1 = {'a': 1, 'b': 2}
    dict2 = {'b': 3, 'c': 4}
    result = merge_dicts(dict1, dict2)
    assert result == {'a': 1, 'b': 5, 'c': 4}

def merge_dicts(dict1, dict2):
    merged = dict1.copy()
    for key, value in dict2.items():
        if key in merged:
            merged[key] += value
        else:
            merged[key] = value
    return merged

# 验证
test_merge_dicts()
print("所有断言通过")

所有断言通过


## 7.5 基于Ollama的大模型本地部署

```bash
# 安装验证
ollama --version
# 部署模型（首次执行自动拉取权重）
ollama run deepseek-r1:8b
# 对话结束后
/bye
```

| 命令 | 说明 |
|---|---|
| `ollama run <model>` | 运行指定模型并启动对话 |
| `ollama pull <model>` | 下载最新模型到本地 |
| `ollama list` | 列出本地所有已下载的模型 |
| `ollama rm <model>` | 删除本地模型 |
| `ollama ps / stop <model>` | 查看/停止正在运行的模型 |

图形界面推荐 **Chatbox**：连接Ollama本地API（默认端口11434），五步完成配置后即可像网页版大模型一样对话。